In [12]:
import json
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path
from collections import Counter

In [8]:
# =========================
# 1) Cấu hình đường dẫn
# =========================
PATH_THANH  = Path("./Thanh.json")
PATH_TRANG  = Path("./Trang.json")
PATH_TRUONG = Path("./Truong.json")

In [9]:
CATEGORIES  = ["positive", "negative", "neutral"]
N_RATERS    = 3

In [13]:
# =========================
# 1) Hàm tiện ích
# =========================
def load_json_flex(path: Path):
    """Đọc JSON: hỗ trợ list-of-dicts hoặc JSON Lines."""
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = [data]
        return data
    except json.JSONDecodeError:
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return rows

def normalize_label(x: str):
    """Chuẩn hoá nhãn về {positive, negative, neutral}."""
    if x is None:
        return None
    s = str(x).strip().lower()
    mapping = {
        "pos": "positive", "positive": "positive", "+": "positive",
        "neg": "negative", "negative": "negative", "-": "negative",
        "neu": "neutral",  "neutral": "neutral",  "0": "neutral"
    }
    return mapping.get(s, None)

def canonicalize_text(t: str):
    """Chuẩn hoá text: NFC, strip, thay nhiều khoảng trắng bằng 1 khoảng."""
    if t is None:
        return None
    # Chuẩn hoá Unicode NFC để so khớp bền vững dấu tiếng Việt
    t = unicodedata.normalize("NFC", str(t))
    # Thay các xuống dòng/tab bằng khoảng trắng và nén khoảng trắng
    t = " ".join(t.split())
    return t

def detect_text_key(rows):
    """Đoán cột text phổ biến."""
    if not rows:
        return None
    keys = set()
    for r in rows:
        if isinstance(r, dict):
            keys.update(r.keys())
    # Ưu tiên 'text', sau đó một số khoá thường gặp
    for k in ["text", "content", "body", "sentence", "review"]:
        if k in keys:
            return k
    # Thử ghép title + content nếu có
    if "title" in keys and "content" in keys:
        return ("title", "content")
    return None

def to_df(rows, annotator_name: str):
    """Chuyển list[dict] -> DataFrame với cột: text_raw, text_canon, label."""
    if len(rows) == 0:
        return pd.DataFrame(columns=["text_raw", "text_canon", annotator_name])

    text_key = detect_text_key(rows)
    # Xác định cột nhãn có thể có
    candidate_label_keys = ["label", "sentiment", "y", "tag", "prediction"]

    # Suy ra label_key
    keys = set()
    for r in rows:
        if isinstance(r, dict):
            keys.update(r.keys())
    label_key = next((k for k in candidate_label_keys if k in keys), None)
    if label_key is None:
        # Không tìm thấy cột nhãn -> None
        label_key = None

    df = pd.DataFrame(rows)

    # Lấy text_raw
    if isinstance(text_key, tuple):
        t1, t2 = text_key
        df["text_raw"] = (df.get(t1, "").astype(str) + " " + df.get(t2, "").astype(str)).str.strip()
    elif text_key is None:
        # Không xác định được text -> dùng repr của dòng (không khuyến nghị)
        df["text_raw"] = df.astype(str).agg(" ".join, axis=1)
    else:
        df["text_raw"] = df[text_key].astype(str)

    df["text_canon"] = df["text_raw"].apply(canonicalize_text)

    # Lấy nhãn
    if label_key is None:
        df[annotator_name] = None
    else:
        df[annotator_name] = df[label_key].apply(normalize_label)

    return df[["text_raw", "text_canon", annotator_name]]

def pick_mode_or_nan(series):
    """Chọn mode của một Series; nếu hòa (nhiều mode) -> NaN."""
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return np.nan
    cnt = Counter(vals)
    most_common = cnt.most_common()
    if len(most_common) == 1:
        return most_common[0][0]
    # kiểm tra hoà
    if len(most_common) >= 2 and most_common[0][1] == most_common[1][1]:
        return np.nan
    return most_common[0][0]

def fleiss_kappa(counts: np.ndarray) -> float:
    """
    counts: (n_items, k) với counts[i, j] = số người gán cho item i ở lớp j.
    """
    n, k = counts.shape
    N = np.sum(counts[0])  # số annotator mỗi item (phải bằng nhau)
    p_j = counts.sum(axis=0) / (n * N)                         # tỉ lệ theo lớp
    P_i = (np.sum(counts**2, axis=1) - N) / (N * (N - 1))      # đồng thuận từng item
    P_bar = np.mean(P_i)
    P_e = np.sum(p_j**2)
    if np.isclose(1 - P_e, 0):
        return np.nan
    return (P_bar - P_e) / (1 - P_e)

In [14]:
# =========================
# 2) Đọc & chuẩn hoá từng annotator
# =========================
rows_thanh  = load_json_flex(PATH_THANH)
rows_trang  = load_json_flex(PATH_TRANG)
rows_truong = load_json_flex(PATH_TRUONG)

df_thanh  = to_df(rows_thanh,  "Thanh")
df_trang  = to_df(rows_trang,  "Trang")
df_truong = to_df(rows_truong, "Truong")

In [15]:
df_trang

,text_raw,text_canon,Trang
0,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,positive
1,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...,neutral
2,Đề thi thử tốt nghiệp THPT môn Toán của thành ...,Đề thi thử tốt nghiệp THPT môn Toán của thành ...,neutral
3,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,neutral
4,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,negative
...,...,...,...
195,"Ai là ‘vua vũ khí’, anh hùng lao động trí óc đ...","Ai là ‘vua vũ khí’, anh hùng lao động trí óc đ...",neutral
196,"Bố mất khi mới lọt lòng, mẹ đi lấy chồng, 2 ch...","Bố mất khi mới lọt lòng, mẹ đi lấy chồng, 2 ch...",negative
197,Bạn đọc giúp đỡ em Nguyễn Hà Phương bị suy thậ...,Bạn đọc giúp đỡ em Nguyễn Hà Phương bị suy thậ...,negative
198,35 dự án ở Quảng Nam nợ hơn 2.000 tỷ đồng tiền...,35 dự án ở Quảng Nam nợ hơn 2.000 tỷ đồng tiền...,negative


In [16]:
df_thanh

,text_raw,text_canon,Thanh
0,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,positive
1,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,neutral
2,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,negative
3,"Tốt nghiệp loại giỏi dù không biết đọc viết, n...","Tốt nghiệp loại giỏi dù không biết đọc viết, n...",negative
4,Màn ‘hỏi xoáy’ bất ngờ của học sinh BRIS với g...,Màn ‘hỏi xoáy’ bất ngờ của học sinh BRIS với g...,positive
...,...,...,...
197,Lý do Ban quản trị chung cư ở TPHCM bị thuế ph...,Lý do Ban quản trị chung cư ở TPHCM bị thuế ph...,negative
198,"Vị trí đắc địa, phong thủy hài hòa của dự án S...","Vị trí đắc địa, phong thủy hài hòa của dự án S...",positive
199,Giá nhà thấp tầng Hà Nội giảm nhưng vẫn gần 30...,Giá nhà thấp tầng Hà Nội giảm nhưng vẫn gần 30...,negative
200,Masterise Homes và những dự án hàng hiệu vươn ...,Masterise Homes và những dự án hàng hiệu vươn ...,positive


In [17]:
df_truong

,text_raw,text_canon,Truong
0,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,positive
1,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...,positive
2,Đề thi thử tốt nghiệp THPT môn Toán của thành ...,Đề thi thử tốt nghiệp THPT môn Toán của thành ...,neutral
3,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,neutral
4,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,negative
...,...,...,...
195,"Ai là ‘vua vũ khí’, anh hùng lao động trí óc đ...","Ai là ‘vua vũ khí’, anh hùng lao động trí óc đ...",positive
196,"Bố mất khi mới lọt lòng, mẹ đi lấy chồng, 2 ch...","Bố mất khi mới lọt lòng, mẹ đi lấy chồng, 2 ch...",negative
197,Bạn đọc giúp đỡ em Nguyễn Hà Phương bị suy thậ...,Bạn đọc giúp đỡ em Nguyễn Hà Phương bị suy thậ...,negative
198,35 dự án ở Quảng Nam nợ hơn 2.000 tỷ đồng tiền...,35 dự án ở Quảng Nam nợ hơn 2.000 tỷ đồng tiền...,negative


In [18]:
# =========================
# 3) Gom theo text_canon trong từng file
#    - Nếu 1 annotator gán nhiều nhãn cho cùng 1 text: lấy mode; nếu hoà -> bỏ (NaN)
# =========================
df_thanh_agg = (df_thanh.groupby(["text_canon"], as_index=False)
                        .agg(text_raw=("text_raw", "first"),
                             Thanh=("Thanh", pick_mode_or_nan)))

df_trang_agg = (df_trang.groupby(["text_canon"], as_index=False)
                        .agg(text_raw=("text_raw", "first"),
                             Trang=("Trang", pick_mode_or_nan)))

df_truong_agg = (df_truong.groupby(["text_canon"], as_index=False)
                          .agg(text_raw=("text_raw", "first"),
                               Truong=("Truong", pick_mode_or_nan)))

In [19]:
# =========================
# 4) Lấy GIAO các text xuất hiện ở cả 3 file
# =========================
common = (df_thanh_agg.merge(df_trang_agg[["text_canon", "Trang"]], on="text_canon", how="inner")
                      .merge(df_truong_agg[["text_canon", "Truong"]], on="text_canon", how="inner"))

# Loại các dòng thiếu nhãn (NaN do hoà hoặc thiếu)
common = common.dropna(subset=["Thanh", "Trang", "Truong"])

print("Số text chung (sau khi lọc đầy đủ 3 nhãn & hoà loại bỏ):", len(common))


Số text chung (sau khi lọc đầy đủ 3 nhãn & hoà loại bỏ): 198


In [20]:
# =========================
# 5) Tạo ma trận đếm cho Fleiss
# =========================
def row_to_counts(row):
    labels = [row["Thanh"], row["Trang"], row["Truong"]]
    return [sum(1 for lb in labels if lb == c) for c in CATEGORIES]

counts_matrix = np.vstack(common.apply(row_to_counts, axis=1).values)

# Kiểm tra tổng phiếu từng dòng
assert np.all(counts_matrix.sum(axis=1) == N_RATERS), "Có dòng tổng phiếu != số annotator!"

In [24]:
# =========================
# 6) Tính Fleiss’s Kappa
# =========================
kappa = fleiss_kappa(counts_matrix)

# =========================
# 7) Báo cáo nhanh
# =========================
print("Fleiss' Kappa (3 annotators, 3 lớp) trên GIAO text:", round(kappa, 4))

# Phân bố đa số để tham khảo
majority_label = [CATEGORIES[np.argmax(row)] for row in counts_matrix]
majority_counts = pd.Series(majority_label).value_counts().reindex(CATEGORIES, fill_value=0)
print("\nPhân bố nhãn theo đa số (trên tập giao):")
print(majority_counts.to_string())

# Tỷ lệ phiếu theo lớp (p_j)
p_j = counts_matrix.sum(axis=0) / counts_matrix.sum()
print("\nTỷ lệ phiếu theo lớp (p_j):")
for cls, pj in zip(CATEGORIES, p_j):
    print(f"  {cls:>8}: {pj:.3f}")

# Nếu cần xuất kết quả chi tiết để kiểm tra:
common.to_csv("./common_texts_with_labels.csv", index=False, encoding="utf-8-sig")
print("Đã lưu chi tiết:", "./common_texts_with_labels.csv")

Fleiss' Kappa (3 annotators, 3 lớp) trên GIAO text: 0.6353

Phân bố nhãn theo đa số (trên tập giao):
positive    51
negative    61
neutral     86

Tỷ lệ phiếu theo lớp (p_j):
  positive: 0.258
  negative: 0.295
   neutral: 0.448
Đã lưu chi tiết: ./common_texts_with_labels.csv


# Trường hợp bất đồng nhau

In [27]:
# Find cases where annotators disagree on labels, and save them to a CSV.
# - Align items by canonicalized text intersection across the three files.
# - Normalize labels to {positive, negative, neutral}.
# - Output rows where at least one annotator differs.
# - Save to ./label_disagreements.csv

import json, unicodedata
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

PATH_THANH  = Path("./Thanh.json")
PATH_TRANG  = Path("./Trang.json")
PATH_TRUONG = Path("./Truong.json")

CATEGORIES = ["positive", "negative", "neutral"]

def load_json_flex(path: Path):
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = [data]
        return data
    except json.JSONDecodeError:
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return rows

def detect_text_key(rows):
    if not rows:
        return None
    keys = set()
    for r in rows:
        if isinstance(r, dict):
            keys.update(r.keys())
    for k in ["text", "content", "body", "sentence", "review"]:
        if k in keys:
            return k
    if "title" in keys and "content" in keys:
        return ("title", "content")
    return None

def canonicalize_text(t: str):
    if t is None:
        return None
    t = unicodedata.normalize("NFC", str(t))
    t = " ".join(t.split())
    return t

def normalize_label(x: str):
    if x is None:
        return None
    s = str(x).strip().lower()
    mapping = {
        "pos": "positive", "positive": "positive", "+": "positive",
        "neg": "negative", "negative": "negative", "-": "negative",
        "neu": "neutral",  "neutral": "neutral",  "0": "neutral"
    }
    return mapping.get(s, None)

def to_df(rows, annotator_name: str):
    if len(rows) == 0:
        return pd.DataFrame(columns=["text_raw", "text_canon", annotator_name])
    text_key = detect_text_key(rows)
    candidate_label_keys = ["label", "sentiment", "y", "tag", "prediction"]
    keys = set()
    for r in rows:
        if isinstance(r, dict):
            keys.update(r.keys())
    label_key = next((k for k in candidate_label_keys if k in keys), None)
    df = pd.DataFrame(rows)
    if isinstance(text_key, tuple):
        t1, t2 = text_key
        df["text_raw"] = (df.get(t1, "").astype(str) + " " + df.get(t2, "").astype(str)).str.strip()
    elif text_key is None:
        df["text_raw"] = df.astype(str).agg(" ".join, axis=1)
    else:
        df["text_raw"] = df[text_key].astype(str)
    df["text_canon"] = df["text_raw"].apply(canonicalize_text)
    if label_key is None:
        df[annotator_name] = None
    else:
        df[annotator_name] = df[label_key].apply(normalize_label)
    return df[["text_raw", "text_canon", annotator_name]]

# Load and convert
rows_thanh  = load_json_flex(PATH_THANH)
rows_trang  = load_json_flex(PATH_TRANG)
rows_truong = load_json_flex(PATH_TRUONG)

df_thanh  = to_df(rows_thanh,  "Thanh")
df_trang  = to_df(rows_trang,  "Trang")
df_truong = to_df(rows_truong, "Truong")

# Aggregate by text_canon for each annotator (if duplicates, pick the most common label; ties -> NaN)
def pick_mode_or_nan(series):
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return np.nan
    c = Counter(vals).most_common()
    if len(c) == 1: 
        return c[0][0]
    if len(c) >= 2 and c[0][1] == c[1][1]:
        return np.nan
    return c[0][0]

agg_thanh  = df_thanh.groupby("text_canon", as_index=False).agg(text_raw=("text_raw", "first"),
                                                                Thanh=("Thanh", pick_mode_or_nan))
agg_trang  = df_trang.groupby("text_canon", as_index=False).agg(text_raw=("text_raw", "first"),
                                                                Trang=("Trang", pick_mode_or_nan))
agg_truong = df_truong.groupby("text_canon", as_index=False).agg(text_raw=("text_raw", "first"),
                                                                 Truong=("Truong", pick_mode_or_nan))

# Intersect texts across all three
common = agg_thanh.merge(agg_trang[["text_canon", "Trang"]], on="text_canon", how="inner") \
                  .merge(agg_truong[["text_canon", "Truong"]], on="text_canon", how="inner")

# Remove rows with any missing label (including ties)
common = common.dropna(subset=["Thanh", "Trang", "Truong"])

# Keep only disagreements
mask_disagree = ~((common["Thanh"] == common["Trang"]) & (common["Trang"] == common["Truong"]))
disagreements = common.loc[mask_disagree].copy()

# Optional: add a quick summary
def majority_vote(row):
    labels = [row["Thanh"], row["Trang"], row["Truong"]]
    cnt = Counter(labels).most_common()
    if len(cnt) == 0:
        return None
    if len(cnt) > 1 and cnt[0][1] == cnt[1][1]:
        return "tie"
    return cnt[0][0]

disagreements["majority_vote"] = disagreements.apply(majority_vote, axis=1)

# Save to CSV
out_path = "./label_disagreements.csv"
disagreements.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"Số văn bản chung (đủ 3 nhãn): {len(common)}")
print(f"Số trường hợp bất đồng nhãn: {len(disagreements)}")
print("Đã lưu CSV:", out_path)

# Print a preview of the first 10 disagreements
print("\n=== Preview 10 disagreements ===")
cols_to_show = ["text_raw", "Thanh", "Trang", "Truong", "majority_vote"]
for i, row in disagreements.head(10).iterrows():
    print("---")
    print("text_raw:", (row["text_raw"][:300] + ("..." if len(row["text_raw"])>300 else "")))
    print("Thanh:", row["Thanh"], "| Trang:", row["Trang"], "| Truong:", row["Truong"], "| majority:", row["majority_vote"])


Số văn bản chung (đủ 3 nhãn): 198
Số trường hợp bất đồng nhãn: 70
Đã lưu CSV: ./label_disagreements.csv

=== Preview 10 disagreements ===
---
text_raw: 'Chúng tôi xung trận không phải để thành anh hùng' — "Chúng tôi còn sống qua chiến tranh là may mắn và cũng là sứ mệnh, sống để tiếp tục cống hiến cho Tổ quốc thay cả phần những đồng đội đã hy sinh" - nữ Anh hùng Phan Thị Ngọc Tươi chia sẻ.
Thanh: positive | Trang: neutral | Truong: neutral | majority: neutral
---
text_raw: 100 ngày đầu nắm quyền đầy 'bão táp' của Tổng thống Mỹ Trump — Trong 100 ngày đầu tiên nắm quyền Tổng thống Mỹ nhiệm kỳ 2, ông Trump đã đưa ra một loạt quyết sách khó đoán trước và làm đảo lộn một số phần của trật tự thế giới.
Thanh: negative | Trang: negative | Truong: neutral | majority: negative
---
text_raw: 18 cô gái bị giam lỏng ở quán karaoke, có em mới 13 tuổi — 18 thiếu nữ 13-20 tuổi bị Dương Phương Thảo thu hết điện thoại, không cho ra ngoài, ép phục vụ ở quán karaoke, massage Moonlight vừa được cảnh sát gi

# Các dòng có trong file file của Trang và Trường mà không có trong file merge (Thanh gán nhầm)

In [28]:
# Re-run: Extract rows present in Truong.json but NOT in the intersection across all three files.
# Outputs:
#  - ./truong_not_in_common_full.csv
#  - ./truong_not_in_common_unique.csv

import json, unicodedata
from pathlib import Path
import pandas as pd

PATH_THANH  = Path("./Thanh.json")
PATH_TRANG  = Path("./Trang.json")
PATH_TRUONG = Path("./Truong.json")

def load_json_flex(path: Path):
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = [data]
        return data
    except json.JSONDecodeError:
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return rows

def detect_text_key(rows):
    if not rows:
        return None
    keys = set()
    for r in rows:
        if isinstance(r, dict):
            keys.update(r.keys())
    for k in ["text", "content", "body", "sentence", "review"]:
        if k in keys:
            return k
    if "title" in keys and "content" in keys:
        return ("title", "content")
    return None

def canonicalize_text(t: str):
    if t is None:
        return None
    t = unicodedata.normalize("NFC", str(t))
    t = " ".join(t.split())
    return t

def to_text_df(rows):
    if len(rows) == 0:
        return pd.DataFrame(columns=["text_raw", "text_canon"])
    key = detect_text_key(rows)
    df = pd.DataFrame(rows)
    if isinstance(key, tuple):
        k1, k2 = key
        df["text_raw"] = (df.get(k1, "").astype(str) + " " + df.get(k2, "").astype(str)).str.strip()
    elif key is None:
        df["text_raw"] = df.astype(str).agg(" ".join, axis=1)
    else:
        df["text_raw"] = df[key].astype(str)
    df["text_canon"] = df["text_raw"].apply(canonicalize_text)
    return df[["text_raw", "text_canon"]]

rows_thanh  = load_json_flex(PATH_THANH)
rows_trang  = load_json_flex(PATH_TRANG)
rows_truong = load_json_flex(PATH_TRUONG)

df_thanh  = to_text_df(rows_thanh)
df_trang  = to_text_df(rows_trang)
df_truong = to_text_df(rows_truong)

set_thanh  = set(df_thanh["text_canon"].dropna().unique())
set_trang  = set(df_trang["text_canon"].dropna().unique())
set_truong = set(df_truong["text_canon"].dropna().unique())

common_texts = set_thanh & set_trang & set_truong

mask_not_in_common = ~df_truong["text_canon"].isin(common_texts)
truong_not_in_common_full = df_truong.loc[mask_not_in_common].copy()

truong_not_in_common_unique = (truong_not_in_common_full
                               .drop_duplicates(subset=["text_canon"], keep="first")
                               .reset_index(drop=True))

out_full   = "./truong_not_in_common_full.csv"
out_unique = "./truong_not_in_common_unique.csv"
truong_not_in_common_full.to_csv(out_full, index=False)
truong_not_in_common_unique.to_csv(out_unique, index=False)

print("=== Summary ===")
print("Thanh unique texts:", len(set_thanh))
print("Trang unique texts:", len(set_trang))
print("Truong unique texts:", len(set_truong))
print("Common texts across all three:", len(common_texts))
print("Truong NOT in common (full, keep duplicates):", len(truong_not_in_common_full))
print("Truong NOT in common (unique by text_canon):", len(truong_not_in_common_unique))
print("\nSaved:")
print(" -", out_full)
print(" -", out_unique)

print("\n=== Preview (first 10 unique) ===")
for i, row in truong_not_in_common_unique.head(10).iterrows():
    snippet = row["text_raw"][:300]
    if len(row["text_raw"]) > 300:
        snippet += "..."
    print(f"[{i}] {snippet}")


=== Summary ===
Thanh unique texts: 202
Trang unique texts: 200
Truong unique texts: 200
Common texts across all three: 198
Truong NOT in common (full, keep duplicates): 2
Truong NOT in common (unique by text_canon): 2

Saved:
 - ./truong_not_in_common_full.csv
 - ./truong_not_in_common_unique.csv

=== Preview (first 10 unique) ===
[0] Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên cứu mạnh thúc đẩy đổi mới sáng tạo — Ngày 18/4 Trường ĐH Tôn Đức Thắng (TDTU) công bố thành lập thêm 4 nhóm nghiên cứu mạnh để bổ sung lực lượng nòng cốt phát triển khoa học công nghệ, đổi mới sáng tạo và chuyển đổi số theo Chiến lược phát triển giáo dục...
[1] Đề thi thử tốt nghiệp THPT môn Toán của thành phố Huế năm 2025 — VietNamNet cập nhật đề thi thử tốt nghiệp THPT năm 2025 môn Toán và đáp án của thành phố Huế để học sinh và thầy cô tham khảo.
